# **0. MNIST 데이터 읽고, 70% 학습 / 30% 테스트 데이터로 나누기**

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform = transforms.ToTensor()

dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

total_size = len(dataset)
train_size = int(0.7 * total_size)
test_size = total_size - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Train 데이터 개수: {len(train_dataset)}개")
print(f"Test 데이터 개수: {len(test_dataset)}개")


Train 데이터 개수: 42000개
Test 데이터 개수: 18000개


***전체 60,000개 중 70%를 학습 데이터, 30%를 테스트 데이터로 사용***

# **1. Hidden Layer가 2개이고, 각 노드가 20개인 MLP 모델:**

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 20)
        self.fc2 = nn.Linear(20, 20)
        self.fc3 = nn.Linear(20, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


*모델 구조*
 - 입력층 : 28*28 = 784
 - 은닉층 1 : 20개 노드
 - 은닉층 2 : 20개 노드
 - 출력층 : 10개 노드

# **2. ReLU와 Sigmoid 비교 MLP 모델 정의**

In [ ]:
class MLP_ReLU(nn.Module):
    def __init__(self):
        super(MLP_ReLU, self).__init__()
        self.fc1 = nn.Linear(28*28, 20)
        self.fc2 = nn.Linear(20, 20)
        self.fc3 = nn.Linear(20, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

class MLP_Sigmoid(nn.Module):
    def __init__(self):
        super(MLP_Sigmoid, self).__init__()
        self.fc1 = nn.Linear(28*28, 20)
        self.fc2 = nn.Linear(20, 20)
        self.fc3 = nn.Linear(20, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.sigmoid(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        x = self.fc3(x)
        return x


*학습 및 테스트 함수(이어지는 항목 공통)*

In [ ]:
def train(model, train_loader, optimizer, criterion, device):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


def test(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    return accuracy


*ReLU, Sigmoid 모델 학습 및 비교 실행*

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

# ReLU 모델 학습
model_relu = MLP_ReLU().to(device)
optimizer_relu = optim.Adam(model_relu.parameters(), lr=0.001)

for epoch in range(5):
    train(model_relu, train_loader, optimizer_relu, criterion, device)

acc_relu = test(model_relu, test_loader, device)
print(f"ReLU 모델 정확도: {acc_relu:.2f}%")

# Sigmoid 모델 학습
model_sigmoid = MLP_Sigmoid().to(device)
optimizer_sigmoid = optim.Adam(model_sigmoid.parameters(), lr=0.001)

for epoch in range(5):
    train(model_sigmoid, train_loader, optimizer_sigmoid, criterion, device)

acc_sigmoid = test(model_sigmoid, test_loader, device)
print(f"Sigmoid 모델 정확도: {acc_sigmoid:.2f}%")


ReLU 모델 정확도: 94.44%
Sigmoid 모델 정확도: 91.82%


***ReLU와 Sigmoid 중 ReLU 모델이 정확도가 높음***

# **3. Batch Normalization 적용 여부에 따른 모델 정의**

In [ ]:
# BN이 없는 MLP 모델
class MLP_NoBN(nn.Module):
    def __init__(self):
        super(MLP_NoBN, self).__init__()
        self.fc1 = nn.Linear(28*28, 20)
        self.fc2 = nn.Linear(20, 20)
        self.fc3 = nn.Linear(20, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# BN이 적용된 MLP 모델
class MLP_BN(nn.Module):
    def __init__(self):
        super(MLP_BN, self).__init__()
        self.fc1 = nn.Linear(28*28, 20)
        self.bn1 = nn.BatchNorm1d(20)
        self.fc2 = nn.Linear(20, 20)
        self.bn2 = nn.BatchNorm1d(20)
        self.fc3 = nn.Linear(20, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return x


In [ ]:
# BN이 없는 MLP 모델 학습
model_no_bn = MLP_NoBN().to(device)
optimizer_no_bn = optim.Adam(model_no_bn.parameters(), lr=0.001)

for epoch in range(5):
    train(model_no_bn, train_loader, optimizer_no_bn, criterion, device)

acc_no_bn = test(model_no_bn, test_loader, device)
print(f"BatchNorm 없는 모델 정확도: {acc_no_bn:.2f}%")


# BN이 있는 MLP 모델 학습
model_bn = MLP_BN().to(device)
optimizer_bn = optim.Adam(model_bn.parameters(), lr=0.001)

for epoch in range(5):
    train(model_bn, train_loader, optimizer_bn, criterion, device)

acc_bn = test(model_bn, test_loader, device)
print(f"BatchNorm 적용된 모델 정확도: {acc_bn:.2f}%")


BatchNorm 없는 모델 정확도: 93.66%
BatchNorm 적용된 모델 정확도: 95.49%


***BatchNorm 적용된 모델의 정확도가 높음***

# **4. 세 가지 가중치(카이밍, 제이비어, 정규분포) 초기화 방식 비교**

In [ ]:
class MLP_Init(nn.Module):
    def __init__(self):
        super(MLP_Init, self).__init__()
        self.fc1 = nn.Linear(28*28, 20)
        self.fc2 = nn.Linear(20, 20)
        self.fc3 = nn.Linear(20, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
def init_weights(model, mode='kaiming'):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if mode == 'kaiming':
                nn.init.kaiming_normal_(m.weight)
            elif mode == 'xavier':
                nn.init.xavier_normal_(m.weight)
            elif mode == 'normal':
                nn.init.normal_(m.weight, mean=0.0, std=0.01)

In [ ]:
# 1. 카이밍 초기화
model_k = MLP_Init().to(device)
init_weights(model_k, mode='kaiming')
optimizer_k = optim.Adam(model_k.parameters(), lr=0.001)

for epoch in range(5):
    train(model_k, train_loader, optimizer_k, criterion, device)

acc_k = test(model_k, test_loader, device)
print(f"카이밍 초기화 정확도: {acc_k:.2f}%")

# 2. 제이비어 초기화
model_x = MLP_Init().to(device)
init_weights(model_x, mode='xavier')
optimizer_x = optim.Adam(model_x.parameters(), lr=0.001)

for epoch in range(5):
    train(model_x, train_loader, optimizer_x, criterion, device)

acc_x = test(model_x, test_loader, device)
print(f"제이비어 초기화 정확도: {acc_x:.2f}%")

# 3. 정규분포 초기화
model_n = MLP_Init().to(device)
init_weights(model_n, mode='normal')
optimizer_n = optim.Adam(model_n.parameters(), lr=0.001)

for epoch in range(5):
    train(model_n, train_loader, optimizer_n, criterion, device)

acc_n = test(model_n, test_loader, device)
print(f"정규분포 초기화 정확도: {acc_n:.2f}%")


카이밍 초기화 정확도: 94.95%
제이비어 초기화 정확도: 94.57%
정규분포 초기화 정확도: 89.76%


***카이밍 초기화 정확도가 제일 높음***

## **# 결론**
---

*가장 정확도가 높은 모델은 활성화 함수가 ReLU이며 BatchNorm을 적용시키고, 카이밍초기화를 했을때 가장 정확도가 높음*